<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/rag_evaluation_scope/RAG_Medical_Assistant_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Address Github Compatibility for nbformat

In [ ]:
import json

with open('RAG_Medical_Assistant_Evaluation.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    for widget_key in nb['metadata']['widgets']:
        if 'state' not in nb['metadata']['widgets'][widget_key]:
            nb['metadata']['widgets'][widget_key]['state'] = {}

with open('RAG_Medical_Assistant_Evaluation.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 105.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install --upgrade huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1  sentence-transformers==5.1.1  -q

In [ ]:
pip install diskcache llama-cpp-python==0.2.28 --no-deps --no-cache-dir -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 191.3 MB/s eta 0:00:00


In [ ]:
!pip install faiss-gpu-cu11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 45.7 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd
import numpy as np

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
import faiss

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
import textwrap
import warnings
warnings.filterwarnings('ignore')

#### function to pretty print a collection

In [ ]:
def pretty_print_doc_collection(relevant_document_chunks):
  for i, chunk in enumerate(relevant_document_chunks):
    print(f"── Chunk {i+1} ──────────────────────────────────────────")
    print(f"Page   : {chunk['metadata'].get('page', 'N/A')}")
    print(f"Score  : {chunk['score']:.4f}")
    print(f"Text   :")
    print(textwrap.fill(chunk['text'], width=80))
    print()

## Basic LLM Response evalution function

In [ ]:
def evaluate_response(response):
    return {
        "word_count":        len(response.split()),
        "has_disclaimer":    any(w in response.lower() for w in ["consult", "disclaimer"]),
        "is_structured":     any(c in response for c in ["1.", "•", "-", "\n"]),
        "mentions_treatment": any(w in response.lower() for w in ["intervention", "management", "treatment","therapy","medication","dose"]),
        "mentions_symptom":   any(w in response.lower() for w in ["recognition", "symptom","sign","present","diagnos"]),
    }

## **Generation**

### **Design Decision: Choosing Generation Model (Decoder)**

In the generation part of Retrieval-Augmented Generation (RAG), an LLM (Large Language Model) is needed because it effectively understands complex queries and generates coherent, context-aware responses in natural language. It seamlessly integrates and synthesizes the retrieved context with the user's query, ensuring accurate and relevant answers. LLMs handle ambiguous or complex questions by reasoning over the retrieved knowledge, generating fluent and human-like text for a better user experience.



| Model | Parameters | Domain Focus | Strengths | Limitations | Best Use Case |
|-------|------------|--------------|-----------|-------------|---------------|
| `epfl-llm/meditron-7b` | ~7B | Medical-adapted | Pretrained on medical corpora (PubMed + clinical guidelines), better on medical QA tasks than base LLaMA | Not instruction-tuned by default; may require fine-tuning or instruction wrapping; advisory warns against direct clinical deployment without alignment :contentReference[oaicite:0]{index=0} | Medical reasoning and domain-specific knowledge when combined with instruction tuning |
| `mistralai/Mistral-7B-Instruct-v0.2` | ~7B | General-purpose instruction | Strong performance per parameter vs similar models; efficient and capable with long context workflows; good conversational abilities :contentReference[oaicite:1]{index=1} | Not specialized for medical domain; lacks built-in safety/moderation; broad rather than deep clinical knowledge | Fast general assistant, good for broader tasks and interactive QA |
| `meta-llama/Meta-Llama-3-8B-Instruct` | ~8B | General instruction | Larger model and newer architecture with instruction fine-tuning; robust general reasoning and dialogue :contentReference[oaicite:2]{index=2} | Still general-purpose; moderate context length compared to some others; may require more resources | General RAG generation with stronger reasoning than typical 7B models |
| `TheBloke/Llama-2-13B-chat-GGUF` | ~13B | Chat optimized | Larger size gives stronger knowledge and coherent chat outputs; fine-tuned for dialogue :contentReference[oaicite:3]{index=3} | Larger and slower than 7B models; not domain-specific; requires more memory/cpu/gpu | Highest-quality open-source chat for broad contexts if resources allow |



We will choose meta-llama/Meta-Llama-3-8B-Instruct which is same as the model we initially started with ("bartowski/Meta-Llama-3-8B-Instruct-GGUF") just in a different format.

The model is selected based on:

* Very strong reasoning ability

* Excellent instruction following

* Handles complex explanations well

Model is suppose to perform well without medical specialization as
medical knowledge exists in general training data, reasoning quality compensates for lack of specialization
and it works very well with RAG and prompt engineering.

### **Design Decision: Context Window**

Llama allows context window of 8192 tokens. We need max 5000 tokens for k=3 to 5 for chunk size of 512

(num_chunks × chunk_size) + system_prompt + question + max_new_tokens < context_window

In [ ]:
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


# Question Answering using RAG

### System and User Prompt Template

In [ ]:
qna_system_message = """<|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>
"""

In [ ]:
qna_user_message_template = """<|start_header_id|>user<|end_header_id|>
<context>
#context
</context>
<question>
#question
</question>
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

### Response Function

In [ ]:
def create_prompt(relevant_document_chunks, system_prompt, user_prompt_template, user_input):

    context_for_query = ". ".join([chunk["text"] for chunk in relevant_document_chunks])

    user_message = user_prompt_template.replace('#context', context_for_query)
    user_message = user_message.replace('#question', user_input)

    prompt = system_prompt + '\n' + user_message

    return prompt

In [ ]:
def generate_rag_response(prompt,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k,
                  stop        = ["<|eot_id|>",
                           "<|start_header_id|>",  "\nassistant", "\nuser"]
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection)
pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query1, relevant_document_chunks)

── Chunk 1 ──────────────────────────────────────────
Page   : 2457
Score  : 0.6607
Text   :
Parenteral antibiotics should be given after specimens of blood, body fluids,
and wound sites have been taken for Gram stain and culture. Very prompt empiric
therapy, started immediately after suspecting sepsis, is essential and may be
lifesaving. Antibiotic selection requires an educated guess based on the
suspected source, clinical setting, knowledge or suspicion of causative
organisms and of sensitivity patterns common to that specific inpatient unit,
and previous culture results. One regimen for septic shock of unknown cause is
gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation
cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if
Pseudomonas is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime
plus a fluoroquinolone (eg, ciprofloxacin) may be used. Monotherapy with maximal
therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
print(prompt)
print(generate_rag_response(prompt));

<|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
16 - Critical Care Medicine Chapter 222. Approach to the Critically Ill Patient Critical care medicine specializes in caring for the most seriously ill patients. These patients are best treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high nurse:patient ratio to provide the necessary h

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query2, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query2, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query2)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
without delay. Contrastenhanced CT has reasonable accuracy in diagnosing appendicitis and can also reveal other causes of an acute abdomen. Graded compression ultrasound can usually be done quickly and uses no radiation (of particular concern in children); however, it is occasionally limited by the presence of bowel gas and is less useful for recognizing nonappendiceal causes of pain. Appendicitis remains primaril

Llama.generate: prefix-match hit


**Clinical Explanation**

Appendicitis is characterized by sudden onset of abdominal pain, anorexia, and abdominal tenderness. The diagnosis is primarily clinical, often supplemented by CT or ultrasound.

**Treatment Protocol**

* Treatment of acute appendicitis is open or laparoscopic appendectomy.
* IV fluids and antibiotics are administered before surgery.
* Third-generation cephalosporins are preferred for antibiotic treatment.
* For non-perforated appendicitis, no further antibiotics are required after initial treatment.
* If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or continued for a fixed course, according to the surgeon's preference.

**Cite**

* The Merck Manual of Diagnosis & Therapy, 19th Edition Chapter 11. Acute Abdomen & Surgical Gastroenterology

**Note**

Appendicitis cannot be cured via medicine alone. Surgery is necessary to remove the inflamed appendix. Delayed treatment increases mortal

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query3, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query3, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query3)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
with oral antimalarials, corticosteroids, retinoids, or immunosuppressants. Hair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be different in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium is usually temporary as well and abates after the precipitating agent is eliminated. • Androgenetic alopecia (male-pattern and 

Llama.generate: prefix-match hit


**Clinical Explanation**

Sudden patchy hair loss, also known as alopecia areata, is a common condition characterized by the sudden onset of patchy hair loss on the scalp or other hairy areas of the body. It is thought to be an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers.

**Treatment Protocol**

The treatment for alopecia areata depends on the extent and severity of the hair loss. In most cases, it is a self-limiting condition, meaning it will resolve on its own without treatment. However, there are several treatments available to promote hair regrowth and reduce the appearance of bald patches:

* Topical corticosteroids: applied directly to the affected area to reduce inflammation
* Minoxidil (Rogaine): a topical solution that stimulates hair growth and slows down hair loss
* Finasteride (Propecia): an oral medication that slows down hair loss and promotes hair regrowth
* Phototherapy: exposure to ultraviolet light or

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query4, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query4, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query4)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
Chapter 324. Traumatic Brain Injury Traumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently impairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily CT). Initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more severe injur

Llama.generate: prefix-match hit


**Clinical Explanation**

Traumatic brain injury (TBI) requires immediate attention to ensure a reliable airway and adequate ventilation, oxygenation, and blood pressure. Surgery may be necessary to place monitors for intracranial pressure management, decompress the brain if pressure is increased, or remove intracranial hematomas.

**Treatment Protocol**

* Ensure a reliable airway
* Maintain adequate ventilation, oxygenation, and blood pressure
* Consider surgery to:
	+ Place monitors for intracranial pressure management
	+ Decompress the brain if pressure is increased
	+ Remove intracranial hematomas
* Manage hypotension by giving fluids and vasopressors as needed
* Monitor and treat seizures promptly, especially in patients with significant structural injury or a Glasgow Coma Scale (GCS) score < 10
* Consider prophylactic anticonvulsants for patients with significant structural injury or GCS < 10
* Provide therapeutic hypothermia to suppress bursts of EEG activity
* Monitor and mana

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query5, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query5, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query5)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
from injuring the skin • To seek medical care at once if an odor emanates from within the cast or if a fever, which may indicate Good hygiene is important. A splint (see Fig. 323-1) can be used to immobilize some stable injuries, including some suspected but unproven fractures, sprains, and other injuries that require immobilization for several days or less. A splint allows patients to apply ice and to move more a

Llama.generate: prefix-match hit


**Clinical Explanation**

A fracture is a crack in the bone that can occur due to various mechanisms such as trauma, osteoporosis, or underlying medical conditions. In this case, the person has fractured their leg during a hiking trip. The primary concern is to ensure proper immobilization and management of pain to prevent further injury and promote healing.

**Treatment Protocol**

1. **Immobilization**: A splint can be used to immobilize the injured leg, allowing for rest and reducing the risk of further injury.
2. **Pain Management**: Analgesics may be prescribed to manage pain and discomfort.
3. **Rest**: The person should avoid putting weight on the affected leg and rest it as much as possible.
4. **Ice and Compression**: Ice packs can be applied intermittently for 15-20 minutes, several times a day, to reduce swelling and pain. Compression bandages or splints can also help minimize swelling.
5. **Elevation**: Elevating the injured leg above the level of the heart can help reduce 

### **Observations**

## **Fine-tuning**

Fine tuning the RAG Architecture, involve picking the right model, **optimal chunking**, **optimal retirval** and **generation**. Below are the 3 different combination that we will try with **Deterministic** and **Conservative Hyperparameter** with **Constrainted Prompting** these we will try to achieve optimal results
| Experiment | Chunk Size | Overlap | TopK | Embedding Model      | LLM Model|
|------------|------------|---------|------|----------------------|------------------------|
| 1          | 256        | 32      | 3    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |
| 2          | 512        | 64      | 5    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |
| 3          | 768        | 64      | 5    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |

We will perform above three experiments with **Deterministic** and **Conservative** LLM configuraiton. Since this is Medical Assistant to provide tools to support quick decision-making and enhance efficiency, we will experiment with towards Deterministic settings.


| Strategy | temperature | top_p | top_k | max_tokens | Use Case |
|---|---|---|---|---|---|
| Deterministic | 0 | 0.95 | 10 | 256 | Factual, consistent answers |
| Conservative | 0.1 | 0.9 | 20 | 256 | Slight variation, still safe |


## **Configuration 1**


This is base case for **Deterministic behavior**. Max output token is 256 which can be a challenge as it potentially can truncate response.

In [ ]:
# Chunk PDF Manual
all_chunks_256_32, file_name = split_chunks(path_to_pdf, 256, 32)
collection = get_or_create_collection(all_chunks_256_32, 256,32);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt) # k=3,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
print(response)

Loaded existing collection with 8125 chunks
**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal of treatment is to identify and eliminate the source of infection, provide supportive care, and manage organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's condition, including vital signs, laboratory results, and physical examination.
2. **Empiric Antibiotic Therapy**: Administer broad-spectrum antibiotics based on the suspected source of infection, clinical setting, and knowledge of common pathogens in the unit.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation to maintain adequate blood pressure and perfusion.
4. **Supportive Care**: Manage organ dysfunction by providing mechanical ventilation, renal replacement therapy, and other supportive measures as needed.
5. **Source Control**: Eliminate the source of infection by draining 

## **Configuration 2**

* Notice the temperature=0.1,
* top_p=0.9.
* Chunck and overlap stays same.
* Reranking model and configuration stays same.

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt, k=3,max_tokens=256,temperature=0.1,top_p=0.9,top_k=20)
print(response)

Loaded existing collection with 8125 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal is to identify early signs of sepsis, initiate empiric antibiotic therapy, and provide supportive care to prevent organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Monitor vital signs, including temperature, blood pressure, pulse, and respiratory rate.
2. **Empiric Antibiotic Therapy**: Administer broad-spectrum antibiotics based on the suspected source of infection, clinical setting, and knowledge of common causative organisms.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation to maintain adequate perfusion and prevent organ dysfunction.
4. **Supportive Care**:
	* Monitor and manage blood glucose levels to prevent hyperglycemia or hypoglycemia.
	* Administer corticosteroids for septic shock and acute respiratory distress syndrome (ARDS).
	* Provide mechanical ventilation as needed for respiratory fai

## **Configuration 3**

* Notice the shift to Deterministic setting temperature=0,top_p=0.95.
* Chunks and Overlap increases to 768 and 64

In [ ]:
all_chunks_768_64, file_name = split_chunks(path_to_pdf, 768, 64)
collection = get_or_create_collection(all_chunks_768_64,768, 64);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt) # k=3,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
print(response)

Loaded existing collection with 8125 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal is to identify early signs of sepsis, initiate empiric antibiotic therapy, and provide supportive care to prevent organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Monitor vital signs, including temperature, blood pressure, pulse, and respiratory rate.
2. **Empiric Antibiotic Therapy**: Administer broad-spectrum antibiotics based on the suspected source of infection, clinical setting, and knowledge of common causative organisms.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation to maintain adequate perfusion and prevent organ dysfunction.
4. **Supportive Care**:
	* Monitor and manage blood glucose levels to prevent hyperglycemia or hypoglycemia.
	* Administer corticosteroids for septic shock, if necessary.
	* Consider activated protein C therapy in patients with severe sepsis and septic shock.
5. **So

## **Configuration 4**

* Notice the shift to Conservative setting temperature=0.1,top_p=0.9
* Chunks and Overlap stays same to 768 and 64
* Top K stays at 10
* Top N (After Reranking) stays at 3
* Lower number compensate for higher token size of 512

In [ ]:
## Using same collection for chunks_size=768, overlap=64
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt,max_tokens=512,temperature=0.1,top_p=0.9,top_k=20)
print(response)

Loaded existing collection with 8125 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal of treatment is to identify and eliminate the source of infection, provide supportive care, and manage organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's clinical status, including vital signs, laboratory results, and imaging studies.
2. **Antibiotic Therapy**: Administer empiric antibiotics based on the suspected source of infection, clinical presentation, and local antibiotic resistance patterns.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation with crystalloid or colloid solutions to maintain adequate blood pressure and perfusion.
4. **Supportive Care**: Manage organ dysfunction by providing mechanical ventilation, vasopressors, and other supportive measures as needed.
5. **Source Control**: Eliminate the source of infection by draining abscesses, removing infected devi

## **Configuration 5**

* Chunk size: 512
* Overlap: 64
* Embedding model: all-mpnet-base-v2
* Vector DB: ChromaDB
* LLM: Llama-3-8B-Instruct (GGUF)
* Temperature: 0
* Max tokens: 512
* Prompt: grounded medical system prompt

In [ ]:
all_chunks_512_64, file_name = split_chunks(path_to_pdf, 512, 64)
collection = get_or_create_collection(all_chunks_512_64, 512,64);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks1 = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks1,qna_system_message,qna_user_message_template, Query1)
Query1_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=3)
print(Query1_response)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In a critical care unit, the protocol for managing sepsis involves a multi-faceted approach that includes:

* Aggressive fluid resuscitation with 0.9% normal saline to maintain adequate blood pressure and perfusion
* Broad-spectrum antibiotics administered empirically based on suspected source of infection and clinical setting
* Drainage of abscesses and excision of necrotic tissue to eliminate septic foci
* Normalization of blood glucose levels through continuous IV insulin infusion
* Replacement-dose corticosteroids to support adrenal function

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's condition, including vital signs, laboratory results, and clinical presentation.
2. **Fluid Resuscitation**: Administer 0.9% normal saline at a rate of 500-1000 mL/h to maintain adequate blood pressure and perfusion.
3. **Antibiotic Therapy**: Initiate 

| Configuration | Name | Chunk Size | Overlap | Temperature | top_p | top_k | max_tokens | Response Quality | Issues |
|---|---|---|---|---|---|---|---|---|---|
| 1 | Small Chunks Deterministic | 256 | 32 | 0 | 0.95 | 10 | 256 | Good structure, covers key protocol steps | Cut off mid-answer, small chunks miss context |
| 2 | Small Chunks Conservative | 256 | 32 | 0.1 | 0.90 | 20 | 256 | More detailed, adds corticosteroids and vasopressors | Cut off mid-sentence |
| 3 | Large Chunks Deterministic | 768 | 64 | 0 | 0.95 | 10 | 256 | Adds activated protein C therapy | Cut off mid-sentence, outdated treatment (withdrawn 2011) |
| 4 | Large Chunks Conservative | 768 | 64 | 0.1 | 0.90 | 20 | 512 | Most complete — cites chapters, blood glucose targets | Stop token bug causes self-evaluation after answer |
| 5 | Medium Chunks Deterministic | 512 | 64 | 0 | 0.95 | 5 | 512 | Balanced chunk size and deterministic generation optimized for accuracy | Best trade-off between context quality, retrieval precision, and response length. |


In [ ]:
# Query2 Response
relevant_document_chunks = chroma_retrieve(Query2, collection)
reranked_chunks2 = rerank(Query2, relevant_document_chunks)
prompt = create_prompt(reranked_chunks2,qna_system_message,qna_user_message_template, Query2)
Query2_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query2_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Appendicitis is characterized by acute inflammation of the vermiform appendix, typically resulting in abdominal pain, anorexia, and abdominal tenderness. The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Additional signs include right lower quadrant direct and rebound tenderness located at McBurney's point, Rovsing sign, psoas sign, and obturator sign.

**Treatment Protocol**

Appendicitis is typically treated with surgical removal of the appendix, either through open or laparoscopic appendectomy. The surgeon can usually remove the appendix even if perforated. IV fluids and antibiotics are administered preoperatively to help manage symptoms and prevent complications.

For nonperforated appendicitis, no further antibiotics are required after surgery. If the appendix is perforated, antibiotics should be continued until t

In [ ]:
# Query3 Response
relevant_document_chunks = chroma_retrieve(Query3, collection)
reranked_chunks3 = rerank(Query3, relevant_document_chunks)
prompt = create_prompt(reranked_chunks3,qna_system_message,qna_user_message_template, Query3)
Query3_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query3_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder characterized by the sudden onset of patchy hair loss on the scalp or other hairy areas. It can occur at any age but is most common in children and young adults.

The possible causes behind alopecia areata include:

* Genetic predisposition
* Environmental triggers (e.g., stress, infection)
* Hormonal changes
* Autoimmune disorders

**Treatment Protocol**

Effective treatments for alopecia areata include:

1. **Topical corticosteroids**: Triamcinolone acetonide suspension can be injected intradermally or potent topical corticosteroids like betamethasone 0.05% bid can be used.
2. **Minoxidil**: Topical minoxidil 1 mL bid applied to the scalp is most effective for vertex alopecia in male-pattern or female-pattern hair loss.
3. **Anthralin**: Topical anthralin (0.5 to 1% for 10 to 20 min daily, then washed off) can be used.
4. **Immunotherapy**: Induction of allergic contact dermati

In [ ]:
# Query4 Response
relevant_document_chunks = chroma_retrieve(Query4, collection)
reranked_chunks4 = rerank(Query4, relevant_document_chunks)
prompt = create_prompt(reranked_chunks4,qna_system_message,qna_user_message_template, Query4)
Query4_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query4_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Traumatic brain injury (TBI) is a physical injury to brain tissue that temporarily or permanently impairs brain function. The initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.

**Treatment Protocol**

* Ensure a reliable airway
* Maintain adequate ventilation, oxygenation, and blood pressure
* Monitor for and manage increased intracranial pressure (ICP)
* Consider surgery in patients with more severe injuries to:
	+ Place monitors to track ICP
	+ Decompress the brain if ICP is increased
	+ Remove intracranial hematomas
* Maintain adequate brain perfusion and oxygenation
* Prevent complications of altered sensorium

**Cite**

Chapter 324. Traumatic Brain Injury, The Merck Ma

In [ ]:
# Query5 Response
relevant_document_chunks = chroma_retrieve(Query5, collection)
reranked_chunks5 = rerank(Query5, relevant_document_chunks)
prompt = create_prompt(reranked_chunks5,qna_system_message,qna_user_message_template, Query5)
Query5_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query5_response)

Llama.generate: prefix-match hit


Clinical Explanation:

A fracture of the leg can occur due to various reasons such as trauma, overuse, or osteoporosis. The symptoms may include pain, swelling, bruising, and limited mobility in the affected limb. It is essential to seek medical attention immediately if you suspect a fracture.

Treatment Protocol:

1. Immobilization: Apply a splint or cast to immobilize the leg and reduce pain.
2. Pain management: Use analgesics such as acetaminophen or NSAIDs to manage pain.
3. Rest: Avoid putting weight on the affected limb and rest it as much as possible.
4. Elevation: Elevate the affected limb above the level of the heart to reduce swelling.
5. Ice application: Apply ice packs to the affected area for 15-20 minutes, several times a day, to reduce pain and inflammation.
6. Compression: Use an elastic bandage or compression wrap to compress the affected area and reduce swelling.
7. Rehabilitation: Gradually increase mobility and strength exercises under the guidance of a healthcare p

# Output Evaluation

We will use the judge model to evaluates:

* Faithfulness – Is the answer supported by retrieved context?
* Relevance – Did it answer the question?
* Medical correctness – Is the treatment aligned with the manual?
* Completeness – Did it miss critical protocol steps?
* Hallucination detection – Did it add unsupported claims?

The right judge model requires strong reasoning, strong instruction following, stable output, not necessarily huge parameter size. It can't be same model as generation model as that will introduce bias. I will use meta-llama/Meta-Llama-3-8B-Instruct because of its stability, performance, strong reasoning and it can perform with Colab T4.

In [ ]:
judge_model_name_or_path="TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
judge_model_basename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"

In [ ]:
judgellm_model_path = hf_hub_download(
    repo_id=judge_model_name_or_path,
    filename=judge_model_basename
)

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
judge_llm = Llama(
    model_path=judgellm_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## Groundedness

Below groundedness rater prompt uses xml tags making them less ambigious. The exmaples (Few-Shot prompt engineering) are provided as reference to model. Model has been told to produce json output in very structured expected format.

We are going to use our knowledge base created from Merck manual to generate an example for prompt.

In [ ]:
groundedness_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Judge whether the <answer> is derived from and supported by the <context>.

Scoring rubric:
- Score 1: Answer contradicts the context or introduces facts not present in context
- Score 2: Answer is loosely related to context but makes unsupported claims
- Score 3: Answer uses context but includes some details not found in context
- Score 4: Answer is mostly grounded in context with minor gaps
- Score 5: Every claim in the answer is directly supported by the context

## Examples

<question>What causes Type 2 diabetes mellitus in adolescents?</question>
<context>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</context>
<answer>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</answer>
{"score": 5, "reason": "Claim directly stated in context.", "unsupported_claims": "none"}

---
<question>What causes Type 2 diabetes?</question>
<context>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</context>
<answer>Type 2 diabetes is caused by sedentary lifestyle.</answer>
{"score": 1, "reason": "Sedentary lifestyle not mentioned in context.", "unsupported_claims": "sedentary lifestyle"}

---
IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
IMPORTANT: Respond ONLY in this exact JSON format, no other text:
{
  "score": <integer 1-5>,
  "reason": "<one sentence explanation>",
  "unsupported_claims": "<list any claims not found in context, or 'none'>"
}
"""

In [ ]:
relevance_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Judge whether the <answer> directly and completely addresses the <question>.

Scoring rubric:
- Score 1: Answer is completely off-topic or does not address the question
- Score 2: Answer addresses the topic but misses the core ask of the question
- Score 3: Answer partially addresses the question but omits key aspects
- Score 4: Answer addresses the question well with minor omissions
- Score 5: Answer directly and completely addresses all aspects of the question

## Examples

<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension can cause headaches and vision problems, and is often asymptomatic.</answer>
{"score": 5, "reason": "Directly answers all aspects of the question.", "missing_aspects": "none"}

---
<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension is a serious cardiovascular condition affecting millions.</answer>
{"score": 1, "reason": "Does not address symptoms at all.", "missing_aspects": "all symptoms"}

---
<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension can cause headaches.</answer>
{"score": 3, "reason": "Mentions headaches but omits vision problems and asymptomatic nature.", "missing_aspects": "vision problems, asymptomatic cases"}

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
IMPORTANT: Respond ONLY in this exact JSON format, no other text:
{
  "score": <integer 1-5>,
  "reason": "<one sentence explanation>",
  "missing_aspects": "<what key aspects of the question were not addressed, or 'none'>"
}
"""

In [ ]:
faithfulness_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Detect whether the <answer> contains hallucinated facts —
claims that are NEITHER supported by the <context> NOR established medical knowledge.

## Examples

<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>The primary treatment for type 2 DM is oral antihyperglycemic drugs</answer>
{"is_faithful": true, "hallucinated_claims": "none", "severity": "none"}

---
<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>Type 2 diabetes is cured by insulin injections taken three times daily.</answer>
{"is_faithful": false, "hallucinated_claims": "insulin injections three times daily, claimed as cure", "severity": "major"}

---
<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>Type 2 diabetes is treated with metformin. Patients should also avoid sugar completely.</answer>
{"is_faithful": false, "hallucinated_claims": "avoid sugar completely", "severity": "minor"}

CRITICAL INSTRUCTIONS:
- Output ONLY one JSON object
- Do NOT change your answer after producing JSON
- Do NOT apologize or revise
- Stop immediately after the closing brace }

...scoring rubric and examples...

IMPORTANT: Respond ONLY in this exact JSON format:
{
  "is_faithful": <true or false>,
  "hallucinated_claims": "<claims or none>",
  "severity": "<none | minor | major>"
}
"""

In [ ]:
# Single template used across all 3 raters
judge_user_message_template = """
<question>{question}</question>
<context>{context}</context>
<answer>{answer}</answer>
"""

In [ ]:
def build_llama3_prompt(system_message, user_message):
    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{system_message}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_message}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>"
    )

In [ ]:
def evaluate_with_judge(judge_llm, question, answer,
                        reranked_chunks, max_tokens=150):
    raters = {
        "groundedness": groundedness_rater_system_message,
        "relevance":    relevance_rater_system_message,
        "faithfulness": faithfulness_rater_system_message,
    }

    context_for_query = ". ".join([chunk["text"] for chunk in reranked_chunks])

    judge_user_message = judge_user_message_template.format(
        question = question,
        context  = context_for_query,
        answer   = answer
    )

    results = {}
    for metric, system_msg in raters.items():
        prompt = build_llama3_prompt(system_msg, judge_user_message)
        raw = judge_llm(
            prompt      = prompt,
            max_tokens  = max_tokens,
            temperature = 0,
            top_p       = 0.95,
            top_k       = 10,
            stop        = [
            "<|eot_id|>",
            "<|start_header_id|>",
            "\nassistant",
            "\nuser",
        ]
        )["choices"][0]["text"].strip()

        # ── Extract first JSON block only ─────────────────────
        try:
            json_start = raw.find("{")
            json_end   = raw.find("}") + 1
            if json_start == -1 or json_end == 0:
                raise ValueError("No JSON found")
            clean = raw[json_start:json_end]
            results[metric] = json.loads(clean)
        except (json.JSONDecodeError, ValueError):
            results[metric] = {"error": "parse_failed", "raw": raw}

    return results

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
evaluate_with_judge(judge_llm,Query1, Query1_response, reranked_chunks1)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'Answer is directly supported by the context.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly answers all aspects of the question, providing a comprehensive protocol for managing sepsis in a critical care unit.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'none',
  'severity': 'minor'}}

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
evaluate_with_judge(judge_llm,Query2, Query2_response, reranked_chunks2)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'Answer is directly supported by the context.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly answers all aspects of the question.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'IV fluids and antibiotics as treatment for non-perforated appendicitis, antibiotics not curative in cases where surgery is impossible',
  'severity': 'minor'}}

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
evaluate_with_judge(judge_llm,Query3, Query3_response, reranked_chunks3)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 3,
  'reason': 'The context does not directly discuss alopecia areata or its causes and treatments. However, the answer mentions some relevant information about this condition.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly addresses the question by providing information on treatments for sudden patchy hair loss and possible causes.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
evaluate_with_judge(judge_llm,Query4, Query4_response, reranked_chunks4)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'The answer directly repeats the information from the context about the recommended treatments for traumatic brain injury.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly and completely addresses the question by detailing the recommended treatments for a person with a brain injury.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
evaluate_with_judge(judge_llm,Query5, Query5_response, reranked_chunks5)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 2,
  'reason': 'Answer is loosely related to context but makes unsupported claims',
  'unsupported_claims': 'The answer does not specifically address a fractured leg during a hiking trip, and some treatment protocols mentioned are not directly applicable to this scenario.'},
 'relevance': {'score': 2,
  'reason': 'The answer partially addresses the question by providing general information about treating various types of injuries and fractures, but does not specifically address a fractured leg during a hiking trip.',
  'missing_aspects': 'Specific treatment protocol for a fractured leg during a hiking trip'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'none',
  'severity': 'minor'}}

| Query | Question | Groundedness | Relevance | Faithfulness | Overall |
|---|---|---|---|---|---|
| Q1 | Managing sepsis in critical care unit | 5/5 | 5/5 | False (minor) | Minor faithfulness concern |
| Q2 | Symptoms & treatment for appendicitis | 5/5 | 5/5 | False (minor) | Minor hallucination detected |
| Q3 | Sudden patchy hair loss treatments | 3/5 | 5/5 | True | Retrieval mismatch |
| Q4 | Treatments for traumatic brain injury | 5/5 | 5/5 | True | Best performing |
| Q5 | Precautions for fractured leg hiking | 2/5 | 2/5 | False (minor) | Retrieval failure |

## Actionable Insights and Business Recommendations

1. Retrieval-Augmented Generation (RAG) significantly improves medical answer accuracy

Experiments show that a standalone LLM without context produces generic responses, occasional hallucinations, missing medical references

After implementing RAG with medical documents (Merck Manuals) responses become more precise answers include domain-specific terminology and hallucination risk is reduced.

**Insight**: Healthcare AI systems must be grounded in trusted medical sources rather than relying solely on pretrained model knowledge.

**Impact**: Higher reliability of clinical information, Reduced risk of misinformation, and better trust from medical professionals

2. Document chunking and embeddings directly affect information retrieval quality

Experiments with chunking and embedding configurations demonstrates that semantic chunking + embedding models allow the system to retrieve relevant medical passages before generating an answer.

Poor chunking can lead to incomplete context, irrelevant retrieval, and weaker answers

**Insight**: The quality of the retrieval layer is as important as the LLM itself.

**Impact**: To build a medical AI assistants we must invest in effort high-quality document preprocessing optimized chunking strategies and domain-specific embedding models

3. Re-Ranking improves context relevance for complex medical queries

By introducing cross-encoder re-ranking, the system filters retrieved chunks to ensure the most relevant medical passages are used. Without re-ranking vector search may retrieve partially relevant documents

With re-ranking, higher contextual precision and more accurate answers can be achieved

**Insight**: Multi-stage retrieval pipelines significantly improve AI reliability for complex knowledge domains like healthcare.

**Impact**: This architecture reduces the chance that incorrect or irrelevant passages influence the final answer.

4. LLM Prompt Engineering Improves Safety and Response Quality

**prompt engineering** helps ensure that the system uses only retrieved context,that hallucinations are minimized and answers follow structured medical explanations

**Insight**: Prompt engineering is a critical control layer for medical AI safety.

**Impact**: Organizations deploying AI assistants should implement strict prompting frameworks that enforce, context grounding, structured answers and source citations

## **Business Recommendations**

1. Deploy AI-Powered Medical knowledge assistants in clinical settings

Healthcare organizations should deploy RAG-based medical assistants that provide quick access to clinical knowledge, summarized treatment guidelines, evidence-based medical information

**Business Value**: Improved physician productivity, Reduced research time, Better patient care outcomes

2. Integrate Medical AI Assistants Into Clinical Workflow Systems

The assistant should be integrated with systems such as Electronic Health Records (EHR), Clinical decision support systems Hospital knowledge portals

**Business Value**: Doctors can retrieve medical information without leaving their workflow environment.

3. Use curated medical knowledge bases instead of open internet sources

Healthcare AI systems should rely on trusted sources such as Merck Manuals

PubMed, Clinical guidelines, peer-reviewed literature

**Business Value**: Improved reliability regulatory compliance and reduced liability risk

4. Implement Multi-Layer safety controls for Medical AI

Before deploying AI in healthcare, organizations should implement grounded RAG architecture, safety prompts, hallucination detection, and  human oversight

**Business Value**: improved regulatory compliance, safer clinical recommendations increased trust from healthcare providers

5. Continuously update the medical knowledge base

Medical guidelines change frequently. The system should support continuous ingestion of new medical literature, automated indexing of updated guidelines

**Business Value**: Healthcare professionals always access to the current medical knowledge.

<font size=6 color='blue'>Power Ahead</font>
___